<a href="https://colab.research.google.com/github/IvanVelezQu/Estructuras-base-de-datos/blob/main/Problemas_hash.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [2]:
import hashlib
import time

def encontrar_secuencia_sha256(hash_objetivo):
    print(f"Iniciando búsqueda para el hash: {hash_objetivo}")
    print("Espacio de búsqueda: 10,000,000,000 combinaciones...")

    tiempo_inicio = time.time()

    # Iteramos desde 0 hasta 9,999,999,999
    for i in range(10000000000):
        # f"{i:010d}" convierte el número en un string de 10 caracteres,
        # rellenando con ceros a la izquierda (ej: 5 -> "0000000005")
        secuencia = f"{i:010d}"

        # Generamos el hash SHA-256
        hash_calculado = hashlib.sha256(secuencia.encode('utf-8')).hexdigest()

        # Comparamos con el objetivo
        if hash_calculado == hash_objetivo:
            tiempo_fin = time.time()
            print(f"\n¡ÉXITO! Secuencia encontrada: {secuencia}")
            print(f"Tiempo de ejecución: {round(tiempo_fin - tiempo_inicio, 2)} segundos")
            return secuencia

        # Imprimir progreso cada 100 millones de intentos para no saturar la consola
        if i > 0 and i % 100000000 == 0:
            porcentaje = (i / 10000000000) * 100
            print(f"Progreso: {porcentaje:.1f}% ({i} hashes calculados...)")

    print("\nBúsqueda finalizada. No se encontró ninguna coincidencia.")
    return None

# --- EJECUCIÓN ---
# Reemplaza este string con tu hash real
MI_HASH_OBJETIVO = "a_aqui_va_el_hash_sha256_de_64_caracteres"

# Asegúrate de pasarlo en minúsculas para evitar errores de comparación
encontrar_secuencia_sha256(MI_HASH_OBJETIVO.lower())

Iniciando búsqueda para el hash: a_aqui_va_tu_hash_sha256_de_64_caracteres
Espacio de búsqueda: 10,000,000,000 combinaciones...


KeyboardInterrupt: 

In [1]:
import hashlib
import itertools

# 1. FUNCIONES BASE DEL ÁRBOL DE MERKLE
def calcular_hash(texto):
    return hashlib.sha256(texto.encode('utf-8')).hexdigest()

def construir_arbol_merkle(transacciones):
    """Construye el árbol y devuelve solo el Hash Raíz."""
    nivel_actual = [calcular_hash(tx) for tx in transacciones]

    while len(nivel_actual) > 1:
        nivel_superior = []
        for i in range(0, len(nivel_actual), 2):
            hoja_izquierda = nivel_actual[i]
            hoja_derecha = nivel_actual[i + 1] if i + 1 < len(nivel_actual) else hoja_izquierda

            nuevo_hash = calcular_hash(hoja_izquierda + hoja_derecha)
            nivel_superior.append(nuevo_hash)

        nivel_actual = nivel_superior

    return nivel_actual[0]

#   MOTOR DE BÚSQUEDA
def encontrar_orden_correcto(root_objetivo, transacciones_desordenadas):
    print(f"Buscando el orden para el Root: {root_objetivo[:15]}...")

    # itertools.permutations genera todas las combinaciones posibles de la lista
    todas_las_combinaciones = itertools.permutations(transacciones_desordenadas)

    intentos = 0
    # Evaluamos cada orden posible
    for orden_actual in todas_las_combinaciones:
        intentos += 1

        # Construimos el árbol con este orden específico
        root_calculado = construir_arbol_merkle(orden_actual)

        # Si coincide, se encontro la resppuesta
        if root_calculado == root_objetivo:
            print(f"\n¡ÉXITO! Orden encontrado tras {intentos} intentos.")
            return list(orden_actual)

    print("\nFallo: Ninguna permutación generó el Hash Raíz objetivo.")
    return None

# EJECUCIÓN DE PRUEBA

# 1. Definimos las transacciones (conocidas, pero digamos que no sabemos el orden real)
transacciones_conocidas = ["Tx_A", "Tx_B", "Tx_C", "Tx_D"]

# (Simulación secreta para obtener un root válido para probar)
# Supongamos que el orden real secreto que generó el root fue: C, A, D, B
orden_secreto = ["Tx_C", "Tx_A", "Tx_D", "Tx_B"]
ROOT_OBJETIVO = construir_arbol_merkle(orden_secreto)

# 2. Le pasamos el ROOT y la lista DESORDENADA a nuestro algoritmo
orden_descubierto = encontrar_orden_correcto(ROOT_OBJETIVO, transacciones_conocidas)

print("-" * 40)
print("Orden correcto de las transacciones:")
for i, tx in enumerate(orden_descubierto):
    print(f"Posición {i}: {tx}")

Buscando el orden para el Root: eaacd8242dc7410...

¡ÉXITO! Orden encontrado tras 14 intentos.
----------------------------------------
Orden correcto de las transacciones:
Posición 0: Tx_C
Posición 1: Tx_A
Posición 2: Tx_D
Posición 3: Tx_B
